> **교육 목표** 시계열 분석의 출발점으로, 날짜를 시간 인덱스로 세우고 순서·빈 시간을 바로잡아 리샘플링합니다.
>
> **핵심 내용** 날짜 변환 → 순서 어긋난 곳 찾기 → 시간 기준 바로잡기 → 빈·중복 시간 확인 → 15분→시간→일 리샘플링 → 기간 자르기

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D5_01 시간 인덱스와 리샘플링

> **오늘 질문:** 다음 주 전력 사용량은? 먼저 시간 축부터 바로 세웁니다.

데이터: 4일차 워크숍의 제철소 전력 데이터(2018년, 15분 간격 35,040건)

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

se = pd.read_csv("../../data/steel_energy.csv")
se[["date", "usage_kwh"]].head()

## 1. 날짜 변환과 시간 인덱스 ✍️

날짜가 '일/월/년 시:분' 형식의 글자입니다. `format`을 지정해 변환하고 인덱스로 세웁니다.

In [ ]:
# 힌트: pd.to_datetime(컬럼, format="%d/%m/%Y %H:%M"), 표.set_index("컬럼")

## 2. 순서가 어긋난 곳 찾기 ✍️

첫날의 마지막 몇 줄(94 ~ 97번째)을 봅니다.

In [ ]:
# 힌트: 표.iloc[시작:끝]

**Q1.** 1월 1일 23:45 다음에 무엇이 오나요? `nsm`(자정 이후 초) 값을 보면 이 '00:00'은 언제를 뜻할까요?

> 답:


## 3. 시간 기준 바로잡기 📋

① 00:00 기록은 하루를 더해 '다음 날 00:00'으로, ② 모든 시각에서 15분을 빼서 '구간 시작 시각'으로 통일합니다.

In [ ]:
idx = se.index
midnight = (idx.hour == 0) & (idx.minute == 0)
fixed = idx + pd.to_timedelta(midnight.astype(int), unit="D")   # ① 00:00 → 다음 날 00:00
se.index = fixed - pd.Timedelta(minutes=15)                       # ② 구간 시작 시각으로
se = se.sort_index()
print(f"정렬 OK? {se.index.is_monotonic_increasing} / 기간 {se.index.min()} ~ {se.index.max()}")

## 4. 빈 시간·중복 시간 확인 ✍️

있어야 할 15분 시각을 모두 만들고, 실제 기록 수와 비교합니다.

> 변수 이름: `full`

In [ ]:
# 힌트: pd.date_range(시작, 끝, freq="15min"), 인덱스.duplicated().sum()

## 5. 리샘플링: 15분 → 시간 → 일 ✍️

사용량은 쌓이는 값이므로 **합계**로 묶습니다.

> 변수 이름: `hourly`, `daily`

In [ ]:
# 힌트: 시리즈.resample("h").sum(), resample("D")

## 6. 기간으로 잘라 보기 ✍️

시간 인덱스가 있으면 `loc["시작":"끝"]`으로 기간을 바로 자를 수 있습니다. 3월 둘째 주를 시간 단위로 봅니다.

In [ ]:
# 힌트: 시리즈.loc["2018-03-05":"2018-03-11"].plot()

## 7. 1년 일별 추이 ✍️

일별 사용량과 7일 이동평균을 함께 그립니다.

In [ ]:
# 힌트: 시리즈.rolling(7).mean()

요일별 평균도 봅니다. (0 = 월요일, 6 = 일요일) 📋

In [ ]:
daily.groupby(daily.index.dayofweek).mean().round(0)

## 정리 💬

**Q2.** 일별 그래프와 요일별 평균에서 어떤 패턴이 보이나요? 다음 주를 예측할 때 가장 먼저 고려해야 할 것은?

> 답:
